<a href="https://colab.research.google.com/github/aniray2908/satellite-esg-risk-engine/blob/main/experiments/python/ceri/ceri_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 4 — Corporate Environmental Risk Index (CERI v1)

## Objective

This notebook formalizes a structured, relative environmental exposure score across mining assets using satellite-derived vegetation metrics.

The goal is to transform validated exposure signals into a scalable, cross-asset risk scoring framework.

CERI v1 is designed to be:

- Interpretable
- Statistically grounded
- Modular
- Extensible
- Comparable across assets

This version uses three engineered features:

F1 — Exposure Intensity  
F2 — Vegetation Suppression  
F3 — Exposure Persistence  

All features are normalized across assets before aggregation.


## Why Move From Metrics to Risk Scoring?

Previous phases validated:

- NDVI-based exposure extraction
- Cross-site generalization
- Stable multi-year signals

However, raw metrics are not decision-ready.

Risk modeling requires:

- Feature engineering
- Cross-asset comparability
- Normalization
- Aggregation logic
- Transparent weighting

CERI v1 introduces a structured aggregation layer.


In [1]:
import pandas as pd
import numpy as np

## Data Loading

We load exported multi-year exposure metrics for:

- Carajás Mine (Brazil)
- Gevra Coal Mine (India)

Each dataset contains:

- Year
- Mean NDVI
- Low NDVI Fraction

We combine them into a unified dataframe for cross-asset feature engineering.

In [3]:
carajas_path = "/content/drive/MyDrive/carajas_pit_centered_full_year_v2_1.csv"
gevra_path = "/content/drive/MyDrive/gevra_pit_centered_full_year_v1.csv"

carajas = pd.read_csv(carajas_path)
gevra = pd.read_csv(gevra_path)

carajas["asset"] = "Carajás"
gevra["asset"] = "Gevra"

df = pd.concat([carajas, gevra], ignore_index=True)

df.head()

,system:index,image_count,low_ndvi_fraction,mean_ndvi,year,.geo,asset
0,0,101,0.865651,0.059416,2019,"{""type"":""MultiPoint"",""coordinates"":[]}",Carajás
1,1,84,0.921274,0.058572,2020,"{""type"":""MultiPoint"",""coordinates"":[]}",Carajás
2,2,89,0.974743,0.012760,2021,"{""type"":""MultiPoint"",""coordinates"":[]}",Carajás
3,3,83,0.954150,0.029671,2022,"{""type"":""MultiPoint"",""coordinates"":[]}",Carajás
4,4,95,0.970843,0.015329,2023,"{""type"":""MultiPoint"",""coordinates"":[]}",Carajás


## Feature Engineering

We derive three asset-level features from multi-year observations.

### F1 — Exposure Intensity

Definition:
Mean Low NDVI Fraction (2019–2023)

Interpretation:
Higher values indicate greater dominance of exposed land within the buffer.

---

### F2 — Vegetation Suppression

Definition:
1 − Mean NDVI

Interpretation:
Lower NDVI implies suppressed vegetation.
We invert NDVI to align higher values with higher risk.

---

### F3 — Exposure Persistence

Definition:
Inverse of variance of Low NDVI Fraction.

Interpretation:
Stable exposure across years indicates sustained industrial footprint.
Lower variance → higher persistence → higher risk.


In [4]:
feature_df = df.groupby("asset").agg({
    "low_ndvi_fraction": ["mean", "var"],
    "mean_ndvi": "mean"
})

feature_df.columns = ["F1_exposure_intensity",
                      "low_ndvi_variance",
                      "mean_ndvi"]

feature_df.reset_index(inplace=True)

# F2: Vegetation Suppression
feature_df["F2_vegetation_suppression"] = 1 - feature_df["mean_ndvi"]

# F3: Persistence (inverse normalized variance)
max_var = feature_df["low_ndvi_variance"].max()
min_var = feature_df["low_ndvi_variance"].min()

feature_df["F3_persistence_raw"] = 1 - (
    (feature_df["low_ndvi_variance"] - min_var) /
    (max_var - min_var + 1e-9)
)

feature_df

,asset,F1_exposure_intensity,low_ndvi_variance,mean_ndvi,F2_vegetation_suppression,F3_persistence_raw
0,Carajás,0.937332,0.002050,0.035150,0.964850,5.025716e-07
1,Gevra,0.684294,0.000061,0.193744,0.806256,1.000000e+00


## Cross-Asset Normalization

Since CERI is designed as a relative score, features must be normalized across assets.

We use Min-Max Scaling:

F_norm = (F - min(F)) / (max(F) - min(F))

This ensures:

- All features are scaled between 0 and 1
- Higher values consistently represent higher relative risk
- Features remain interpretable
- Composite aggregation remains transparent

Note:
With only two assets, scaling produces extreme contrasts.
This framework is designed to scale with larger asset sets.


In [5]:
features = ["F1_exposure_intensity",
            "F2_vegetation_suppression",
            "F3_persistence_raw"]

for col in features:
    min_val = feature_df[col].min()
    max_val = feature_df[col].max()
    feature_df[col + "_norm"] = (
        (feature_df[col] - min_val) /
        (max_val - min_val + 1e-9)
    )

feature_df

,asset,F1_exposure_intensity,low_ndvi_variance,mean_ndvi,F2_vegetation_suppression,F3_persistence_raw,F1_exposure_intensity_norm,F2_vegetation_suppression_norm,F3_persistence_raw_norm
0,Carajás,0.937332,0.002050,0.035150,0.964850,5.025716e-07,1.0,1.0,0.0
1,Gevra,0.684294,0.000061,0.193744,0.806256,1.000000e+00,0.0,0.0,1.0


## Composite Score Design — CERI v1

We define CERI as a weighted linear combination:

CERI = w1 * F1_norm + w2 * F2_norm + w3 * F3_norm

Initial weights:

- w1 = 0.5 (Exposure Intensity)
- w2 = 0.3 (Vegetation Suppression)
- w3 = 0.2 (Persistence)

Rationale:

Exposure dominance is considered the primary driver of environmental footprint.
Vegetation suppression reinforces exposure intensity.
Persistence adds stability consideration.

Weights sum to 1 for interpretability.


In [6]:
w1 = 0.5
w2 = 0.3
w3 = 0.2

feature_df["CERI_v1"] = (
    w1 * feature_df["F1_exposure_intensity_norm"] +
    w2 * feature_df["F2_vegetation_suppression_norm"] +
    w3 * feature_df["F3_persistence_raw_norm"]
)

feature_df.sort_values("CERI_v1", ascending=False)

,asset,F1_exposure_intensity,low_ndvi_variance,mean_ndvi,F2_vegetation_suppression,F3_persistence_raw,F1_exposure_intensity_norm,F2_vegetation_suppression_norm,F3_persistence_raw_norm,CERI_v1
0,Carajás,0.937332,0.002050,0.035150,0.964850,5.025716e-07,1.0,1.0,0.0,0.8
1,Gevra,0.684294,0.000061,0.193744,0.806256,1.000000e+00,0.0,0.0,1.0,0.2


## Interpretation

CERI_v1 provides a relative exposure ranking.

The higher the CERI score:

- The greater the exposed land dominance
- The lower the vegetation health
- The more persistent the exposure over time

With two assets, normalization results in maximal separation.

As additional assets are introduced, CERI becomes more distributionally stable and analytically meaningful.


## Weight Sensitivity Analysis

Risk scoring models should not be overly sensitive to arbitrary weights.

We test an alternative weighting scheme to assess ranking stability.


In [7]:
w1_alt = 0.4
w2_alt = 0.4
w3_alt = 0.2

feature_df["CERI_alt"] = (
    w1_alt * feature_df["F1_exposure_intensity_norm"] +
    w2_alt * feature_df["F2_vegetation_suppression_norm"] +
    w3_alt * feature_df["F3_persistence_raw_norm"]
)

feature_df[["asset", "CERI_v1", "CERI_alt"]]

,asset,CERI_v1,CERI_alt
0,Carajás,0.8,0.8
1,Gevra,0.2,0.2


## CERI v1 — Summary

CERI v1 demonstrates:

- Structured feature engineering
- Cross-asset normalization
- Transparent aggregation
- Relative risk ranking
- Extensible architecture

Next evolutionary directions include:

- Z-score normalization
- PCA-based weighting
- Entropy-based feature importance
- Clustering-based risk tiering
- Multi-signal integration

CERI v1 represents the first formal modeling layer
built upon validated satellite-derived environmental exposure signals.





---


---


## CERI v1.1 — Z-Score Normalization

To strengthen statistical grounding, we compute z-scores for each feature.

Z-score normalization measures how many standard deviations an asset lies above or below the population mean.

This provides:

- Distribution-aware scaling
- Improved comparability
- Robustness as asset count increases
- Stronger statistical interpretation

Positive CERI_z values indicate above-average environmental exposure.
Negative values indicate below-average exposure.


In [8]:
z_features = ["F1_exposure_intensity",
              "F2_vegetation_suppression",
              "F3_persistence_raw"]

for col in z_features:
    mean_val = feature_df[col].mean()
    std_val = feature_df[col].std()
    feature_df[col + "_z"] = (
        (feature_df[col] - mean_val) /
        (std_val + 1e-9)
    )

feature_df

,asset,F1_exposure_intensity,low_ndvi_variance,mean_ndvi,F2_vegetation_suppression,F3_persistence_raw,F1_exposure_intensity_norm,F2_vegetation_suppression_norm,F3_persistence_raw_norm,CERI_v1,CERI_alt,F1_exposure_intensity_z,F2_vegetation_suppression_z,F3_persistence_raw_z
0,Carajás,0.937332,0.002050,0.035150,0.964850,5.025716e-07,1.0,1.0,0.0,0.8,0.8,0.707107,0.707107,-0.707107
1,Gevra,0.684294,0.000061,0.193744,0.806256,1.000000e+00,0.0,0.0,1.0,0.2,0.2,-0.707107,-0.707107,0.707107


In [9]:
feature_df["CERI_z"] = (
    w1 * feature_df["F1_exposure_intensity_z"] +
    w2 * feature_df["F2_vegetation_suppression_z"] +
    w3 * feature_df["F3_persistence_raw_z"]
)

feature_df.sort_values("CERI_z", ascending=False)

,asset,F1_exposure_intensity,low_ndvi_variance,mean_ndvi,F2_vegetation_suppression,F3_persistence_raw,F1_exposure_intensity_norm,F2_vegetation_suppression_norm,F3_persistence_raw_norm,CERI_v1,CERI_alt,F1_exposure_intensity_z,F2_vegetation_suppression_z,F3_persistence_raw_z,CERI_z
0,Carajás,0.937332,0.002050,0.035150,0.964850,5.025716e-07,1.0,1.0,0.0,0.8,0.8,0.707107,0.707107,-0.707107,0.424264
1,Gevra,0.684294,0.000061,0.193744,0.806256,1.000000e+00,0.0,0.0,1.0,0.2,0.2,-0.707107,-0.707107,0.707107,-0.424264


## Interpretation — CERI_z

CERI_z reflects relative environmental exposure intensity in standard deviation units.

- CERI_z > 0 → Above-average exposure risk
- CERI_z < 0 → Below-average exposure risk

As additional assets are introduced, this score becomes more stable and statistically meaningful.

This transition from min-max scaling to z-score normalization strengthens the modeling rigor of CERI.



---



---

## CERI v1.2 — Unsupervised Risk Tiering

While CERI_v1 and CERI_z provide continuous risk scores,
decision systems often require categorical risk tiers.

We introduce unsupervised clustering to:

- Segment assets based on exposure characteristics
- Automatically identify relative risk groups
- Demonstrate scalability toward larger asset sets

Clustering is performed on standardized (z-score) features.

Note:
With only two assets, clustering results are trivial.
However, this framework is designed to scale meaningfully as more assets are introduced.


In [10]:
from sklearn.cluster import KMeans

In [11]:
# Use z-score normalized features
cluster_features = feature_df[[
    "F1_exposure_intensity_z",
    "F2_vegetation_suppression_z",
    "F3_persistence_raw_z"
]]

cluster_features

,F1_exposure_intensity_z,F2_vegetation_suppression_z,F3_persistence_raw_z
0,0.707107,0.707107,-0.707107
1,-0.707107,-0.707107,0.707107


In [12]:
kmeans = KMeans(n_clusters=2, random_state=42)
feature_df["risk_cluster"] = kmeans.fit_predict(cluster_features)

feature_df

,asset,F1_exposure_intensity,low_ndvi_variance,mean_ndvi,F2_vegetation_suppression,F3_persistence_raw,F1_exposure_intensity_norm,F2_vegetation_suppression_norm,F3_persistence_raw_norm,CERI_v1,CERI_alt,F1_exposure_intensity_z,F2_vegetation_suppression_z,F3_persistence_raw_z,CERI_z,risk_cluster
0,Carajás,0.937332,0.002050,0.035150,0.964850,5.025716e-07,1.0,1.0,0.0,0.8,0.8,0.707107,0.707107,-0.707107,0.424264,0
1,Gevra,0.684294,0.000061,0.193744,0.806256,1.000000e+00,0.0,0.0,1.0,0.2,0.2,-0.707107,-0.707107,0.707107,-0.424264,1


## Interpreting Clusters

Clusters represent relative exposure groupings.

We map clusters to risk tiers based on mean CERI_z:

- Higher CERI_z → High Risk Tier
- Lower CERI_z → Moderate/Lower Risk Tier

This mapping maintains interpretability while allowing automated segmentation.


In [13]:
# Determine which cluster has higher average CERI_z
cluster_means = feature_df.groupby("risk_cluster")["CERI_z"].mean()

high_risk_cluster = cluster_means.idxmax()

feature_df["risk_tier"] = feature_df["risk_cluster"].apply(
    lambda x: "High Risk" if x == high_risk_cluster else "Moderate Risk"
)

feature_df[["asset", "CERI_z", "risk_tier"]]

,asset,CERI_z,risk_tier
0,Carajás,0.424264,High Risk
1,Gevra,-0.424264,Moderate Risk


## Interpretation — Risk Tiering

The clustering model segments assets based on exposure feature similarity.

Although trivial with two assets, this structure enables:

- Automated risk tiering
- Expansion to multi-asset portfolios
- Removal of manual threshold definitions
- Integration with additional features in future phases

This demonstrates how environmental exposure metrics can evolve into
a semi-automated satellite-derived risk intelligence framework.
